<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_05_window_scaling_seq2one/stage_05_window_scaling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_05_window_scaling**




## **0. Configuración del Entorno**


### 0.1. Acceso a Drive

In [65]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [66]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [67]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib
import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

# ----------------------------
# Logging
# ----------------------------
logging.basicConfig(
    level=os.environ.get("LOG_LEVEL", "INFO"),
    format="%(asctime)s | %(levelname)s | %(message)s",
)
log = logging.getLogger("stage_06_window_scaling_seq2seq")

### 0.4. Definición de rutas

In [68]:
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

#### RUTAS DE ENTRADA

In [69]:
# ============================================================
# DELTA 60
# ============================================================
IN_PARQUET_DELTA_60_TRAIN = Path(os.environ.get("IN_PARQUET_DELTA_60_TRAIN", "data/splits/mnq_delta_60_train.parquet"))
IN_PARQUET_DELTA_60_VALID = Path(os.environ.get("IN_PARQUET_DELTA_60_VALID", "data/splits/mnq_delta_60_valid.parquet"))
IN_PARQUET_DELTA_60_TEST = Path(os.environ.get("IN_PARQUET_DELTA_60_TEST", "data/splits/mnq_delta_60_test.parquet"))

# ============================================================
# DELTA 90
# ============================================================
IN_PARQUET_DELTA_90_TRAIN = Path(os.environ.get("IN_PARQUET_DELTA_90_TRAIN", "data/splits/mnq_delta_90_train.parquet"))
IN_PARQUET_DELTA_90_VALID = Path(os.environ.get("IN_PARQUET_DELTA_90_VALID", "data/splits/mnq_delta_90_valid.parquet"))
IN_PARQUET_DELTA_90_TEST = Path(os.environ.get("IN_PARQUET_DELTA_90_TEST", "data/splits/mnq_delta_90_test.parquet"))

# ============================================================
# RET 60
# ============================================================
IN_PARQUET_RET_60_TRAIN = Path(os.environ.get("IN_PARQUET_RET_60_TRAIN", "data/splits/mnq_ret_60_train.parquet"))
IN_PARQUET_RET_60_VALID = Path(os.environ.get("IN_PARQUET_RET_60_VALID", "data/splits/mnq_ret_60_valid.parquet"))
IN_PARQUET_RET_60_TEST = Path(os.environ.get("IN_PARQUET_RET_60_TEST", "data/splits/mnq_ret_60_test.parquet"))

# ============================================================
# RET 90
# ============================================================
IN_PARQUET_RET_90_TRAIN = Path(os.environ.get("IN_PARQUET_RET_90_TRAIN", "data/splits/mnq_ret_90_train.parquet"))
IN_PARQUET_RET_90_VALID = Path(os.environ.get("IN_PARQUET_RET_90_VALID", "data/splits/mnq_ret_90_valid.parquet"))
IN_PARQUET_RET_90_TEST = Path(os.environ.get("IN_PARQUET_RET_90_TEST", "data/splits/mnq_ret_90_test.parquet"))

In [70]:
IN_PARQUET_DELTA_60_TRAIN = DRIVE_DIR / IN_PARQUET_DELTA_60_TRAIN
IN_PARQUET_DELTA_60_VALID = DRIVE_DIR / IN_PARQUET_DELTA_60_VALID
IN_PARQUET_DELTA_60_TEST = DRIVE_DIR / IN_PARQUET_DELTA_60_TEST

IN_PARQUET_DELTA_90_TRAIN = DRIVE_DIR / IN_PARQUET_DELTA_90_TRAIN
IN_PARQUET_DELTA_90_VALID = DRIVE_DIR / IN_PARQUET_DELTA_90_VALID
IN_PARQUET_DELTA_90_TEST = DRIVE_DIR / IN_PARQUET_DELTA_90_TEST

IN_PARQUET_RET_60_TRAIN = DRIVE_DIR / IN_PARQUET_RET_60_TRAIN
IN_PARQUET_RET_60_VALID = DRIVE_DIR / IN_PARQUET_RET_60_VALID
IN_PARQUET_RET_60_TEST = DRIVE_DIR / IN_PARQUET_RET_60_TEST

IN_PARQUET_RET_90_TRAIN = DRIVE_DIR / IN_PARQUET_RET_90_TRAIN
IN_PARQUET_RET_90_VALID = DRIVE_DIR / IN_PARQUET_RET_90_VALID
IN_PARQUET_RET_90_TEST = DRIVE_DIR / IN_PARQUET_RET_90_TEST

#### RUTAS DE SALIDA DE PARQUETS ESCALADOS

In [71]:
# ============================================================
# DELTA 60
# ============================================================
OUT_PARQUET_DELTA_60_TRAIN_Z = Path(os.environ.get("OUT_PARQUET_DELTA_60_TRAIN_Z", "data/scaled/mnq_delta_60_train_z.parquet"))
OUT_PARQUET_DELTA_60_VALID_Z = Path(os.environ.get("OUT_PARQUET_DELTA_60_VALID_Z", "data/scaled/mnq_delta_60_valid_z.parquet"))
OUT_PARQUET_DELTA_60_TEST_Z  = Path(os.environ.get("OUT_PARQUET_DELTA_60_TEST_Z",  "data/scaled/mnq_delta_60_test_z.parquet"))
OUT_SCALER_DELTA_60          = Path(os.environ.get("OUT_SCALER_DELTA_60",          "data/scaled/scaler_delta_60.pkl"))

# ============================================================
# DELTA 90
# ============================================================
OUT_PARQUET_DELTA_90_TRAIN_Z = Path(os.environ.get("OUT_PARQUET_DELTA_90_TRAIN_Z", "data/scaled/mnq_delta_90_train_z.parquet"))
OUT_PARQUET_DELTA_90_VALID_Z = Path(os.environ.get("OUT_PARQUET_DELTA_90_VALID_Z", "data/scaled/mnq_delta_90_valid_z.parquet"))
OUT_PARQUET_DELTA_90_TEST_Z  = Path(os.environ.get("OUT_PARQUET_DELTA_90_TEST_Z",  "data/scaled/mnq_delta_90_test_z.parquet"))
OUT_SCALER_DELTA_90          = Path(os.environ.get("OUT_SCALER_DELTA_90",          "data/scaled/scaler_delta_90.pkl"))

# ============================================================
# RET 60
# ============================================================
OUT_PARQUET_RET_60_TRAIN_Z = Path(os.environ.get("OUT_PARQUET_RET_60_TRAIN_Z", "data/scaled/mnq_ret_60_train_z.parquet"))
OUT_PARQUET_RET_60_VALID_Z = Path(os.environ.get("OUT_PARQUET_RET_60_VALID_Z", "data/scaled/mnq_ret_60_valid_z.parquet"))
OUT_PARQUET_RET_60_TEST_Z  = Path(os.environ.get("OUT_PARQUET_RET_60_TEST_Z",  "data/scaled/mnq_ret_60_test_z.parquet"))
OUT_SCALER_RET_60          = Path(os.environ.get("OUT_SCALER_RET_60",          "data/scaled/scaler_ret_60.pkl"))

# ============================================================
# RET 90
# ============================================================
OUT_PARQUET_RET_90_TRAIN_Z = Path(os.environ.get("OUT_PARQUET_RET_90_TRAIN_Z", "data/scaled/mnq_ret_90_train_z.parquet"))
OUT_PARQUET_RET_90_VALID_Z = Path(os.environ.get("OUT_PARQUET_RET_90_VALID_Z", "data/scaled/mnq_ret_90_valid_z.parquet"))
OUT_PARQUET_RET_90_TEST_Z  = Path(os.environ.get("OUT_PARQUET_RET_90_TEST_Z",  "data/scaled/mnq_ret_90_test_z.parquet"))
OUT_SCALER_RET_90          = Path(os.environ.get("OUT_SCALER_RET_90",          "data/scaled/scaler_ret_90.pkl"))

In [72]:
OUT_PARQUET_DELTA_60_TRAIN_Z = DRIVE_DIR / OUT_PARQUET_DELTA_60_TRAIN_Z
OUT_PARQUET_DELTA_60_VALID_Z = DRIVE_DIR / OUT_PARQUET_DELTA_60_VALID_Z
OUT_PARQUET_DELTA_60_TEST_Z  = DRIVE_DIR / OUT_PARQUET_DELTA_60_TEST_Z
OUT_SCALER_DELTA_60          = DRIVE_DIR / OUT_SCALER_DELTA_60

OUT_PARQUET_DELTA_90_TRAIN_Z = DRIVE_DIR / OUT_PARQUET_DELTA_90_TRAIN_Z
OUT_PARQUET_DELTA_90_VALID_Z = DRIVE_DIR / OUT_PARQUET_DELTA_90_VALID_Z
OUT_PARQUET_DELTA_90_TEST_Z  = DRIVE_DIR / OUT_PARQUET_DELTA_90_TEST_Z
OUT_SCALER_DELTA_90          = DRIVE_DIR / OUT_SCALER_DELTA_90

OUT_PARQUET_RET_60_TRAIN_Z = DRIVE_DIR / OUT_PARQUET_RET_60_TRAIN_Z
OUT_PARQUET_RET_60_VALID_Z = DRIVE_DIR / OUT_PARQUET_RET_60_VALID_Z
OUT_PARQUET_RET_60_TEST_Z  = DRIVE_DIR / OUT_PARQUET_RET_60_TEST_Z
OUT_SCALER_RET_60          = DRIVE_DIR / OUT_SCALER_RET_60

OUT_PARQUET_RET_90_TRAIN_Z = DRIVE_DIR / OUT_PARQUET_RET_90_TRAIN_Z
OUT_PARQUET_RET_90_VALID_Z = DRIVE_DIR / OUT_PARQUET_RET_90_VALID_Z
OUT_PARQUET_RET_90_TEST_Z  = DRIVE_DIR / OUT_PARQUET_RET_90_TEST_Z
OUT_SCALER_RET_90          = DRIVE_DIR / OUT_SCALER_RET_90

#### RUTAS DE SALIDA DE VENTANAS

In [73]:
from pathlib import Path
import os

# =========================
# SEQ2ONE (NPZ unificado por split)
# =========================

OUT_SEQ2ONE_DELTA_60_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_DELTA_60_TRAIN", "data/windows/seq2one/windows_delta_60_train.npz")
OUT_SEQ2ONE_DELTA_60_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_DELTA_60_VALID", "data/windows/seq2one/windows_delta_60_valid.npz")
OUT_SEQ2ONE_DELTA_60_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_DELTA_60_TEST",  "data/windows/seq2one/windows_delta_60_test.npz")

OUT_SEQ2ONE_DELTA_90_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_DELTA_90_TRAIN", "data/windows/seq2one/windows_delta_90_train.npz")
OUT_SEQ2ONE_DELTA_90_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_DELTA_90_VALID", "data/windows/seq2one/windows_delta_90_valid.npz")
OUT_SEQ2ONE_DELTA_90_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_DELTA_90_TEST",  "data/windows/seq2one/windows_delta_90_test.npz")

OUT_SEQ2ONE_RET_60_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_RET_60_TRAIN", "data/windows/seq2one/windows_ret_60_train.npz")
OUT_SEQ2ONE_RET_60_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_RET_60_VALID", "data/windows/seq2one/windows_ret_60_valid.npz")
OUT_SEQ2ONE_RET_60_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_RET_60_TEST",  "data/windows/seq2one/windows_ret_60_test.npz")

OUT_SEQ2ONE_RET_90_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_RET_90_TRAIN", "data/windows/seq2one/windows_ret_90_train.npz")
OUT_SEQ2ONE_RET_90_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_RET_90_VALID", "data/windows/seq2one/windows_ret_90_valid.npz")
OUT_SEQ2ONE_RET_90_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2ONE_RET_90_TEST",  "data/windows/seq2one/windows_ret_90_test.npz")


# =========================
# SEQ2SEQ (NPZ unificado por split)
# =========================

OUT_SEQ2SEQ_DELTA_60_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_DELTA_60_TRAIN", "data/windows/seq2seq/windows_delta_60_train.npz")
OUT_SEQ2SEQ_DELTA_60_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_DELTA_60_VALID", "data/windows/seq2seq/windows_delta_60_valid.npz")
OUT_SEQ2SEQ_DELTA_60_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_DELTA_60_TEST",  "data/windows/seq2seq/windows_delta_60_test.npz")

OUT_SEQ2SEQ_DELTA_90_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_DELTA_90_TRAIN", "data/windows/seq2seq/windows_delta_90_train.npz")
OUT_SEQ2SEQ_DELTA_90_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_DELTA_90_VALID", "data/windows/seq2seq/windows_delta_90_valid.npz")
OUT_SEQ2SEQ_DELTA_90_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_DELTA_90_TEST",  "data/windows/seq2seq/windows_delta_90_test.npz")

OUT_SEQ2SEQ_RET_60_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_RET_60_TRAIN", "data/windows/seq2seq/windows_ret_60_train.npz")
OUT_SEQ2SEQ_RET_60_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_RET_60_VALID", "data/windows/seq2seq/windows_ret_60_valid.npz")
OUT_SEQ2SEQ_RET_60_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_RET_60_TEST",  "data/windows/seq2seq/windows_ret_60_test.npz")

OUT_SEQ2SEQ_RET_90_TRAIN = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_RET_90_TRAIN", "data/windows/seq2seq/windows_ret_90_train.npz")
OUT_SEQ2SEQ_RET_90_VALID = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_RET_90_VALID", "data/windows/seq2seq/windows_ret_90_valid.npz")
OUT_SEQ2SEQ_RET_90_TEST  = DRIVE_DIR / os.environ.get("OUT_SEQ2SEQ_RET_90_TEST",  "data/windows/seq2seq/windows_ret_90_test.npz")

# **1. Carga de datos**

## 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [74]:
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def load_mnq_parquet(path: Path):
    os.path.exists(path)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(path)
    return mnq_parquet

## **1.2. Información de datasets**


In [75]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")


## **1.3. Carga de mnq e información**


In [76]:
# =========================
# DELTA 60
# =========================
mnq_delta_60_train = load_mnq_parquet(IN_PARQUET_DELTA_60_TRAIN)
info_mnq_delta_60_train = mnq_dataset_info(mnq_delta_60_train, name="mnq_delta_60_train", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_delta_60_train)

mnq_delta_60_valid = load_mnq_parquet(IN_PARQUET_DELTA_60_VALID)
info_mnq_delta_60_valid = mnq_dataset_info(mnq_delta_60_valid, name="mnq_delta_60_valid", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_delta_60_valid)

mnq_delta_60_test = load_mnq_parquet(IN_PARQUET_DELTA_60_TEST)
info_mnq_delta_60_test = mnq_dataset_info(mnq_delta_60_test, name="mnq_delta_60_test", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_delta_60_test)


# =========================
# DELTA 90
# =========================
mnq_delta_90_train = load_mnq_parquet(IN_PARQUET_DELTA_90_TRAIN)
info_mnq_delta_90_train = mnq_dataset_info(mnq_delta_90_train, name="mnq_delta_90_train", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_delta_90_train)

mnq_delta_90_valid = load_mnq_parquet(IN_PARQUET_DELTA_90_VALID)
info_mnq_delta_90_valid = mnq_dataset_info(mnq_delta_90_valid, name="mnq_delta_90_valid", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_delta_90_valid)

mnq_delta_90_test = load_mnq_parquet(IN_PARQUET_DELTA_90_TEST)
info_mnq_delta_90_test = mnq_dataset_info(mnq_delta_90_test, name="mnq_delta_90_test", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_delta_90_test)


# =========================
# RET 60
# =========================
mnq_ret_60_train = load_mnq_parquet(IN_PARQUET_RET_60_TRAIN)
info_mnq_ret_60_train = mnq_dataset_info(mnq_ret_60_train, name="mnq_ret_60_train", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_ret_60_train)

mnq_ret_60_valid = load_mnq_parquet(IN_PARQUET_RET_60_VALID)
info_mnq_ret_60_valid = mnq_dataset_info(mnq_ret_60_valid, name="mnq_ret_60_valid", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_ret_60_valid)

mnq_ret_60_test = load_mnq_parquet(IN_PARQUET_RET_60_TEST)
info_mnq_ret_60_test = mnq_dataset_info(mnq_ret_60_test, name="mnq_ret_60_test", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_ret_60_test)


# =========================
# RET 90
# =========================
mnq_ret_90_train = load_mnq_parquet(IN_PARQUET_RET_90_TRAIN)
info_mnq_ret_90_train = mnq_dataset_info(mnq_ret_90_train, name="mnq_ret_90_train", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_ret_90_train)

mnq_ret_90_valid = load_mnq_parquet(IN_PARQUET_RET_90_VALID)
info_mnq_ret_90_valid = mnq_dataset_info(mnq_ret_90_valid, name="mnq_ret_90_valid", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_ret_90_valid)

mnq_ret_90_test = load_mnq_parquet(IN_PARQUET_RET_90_TEST)
info_mnq_ret_90_test = mnq_dataset_info(mnq_ret_90_test, name="mnq_ret_90_test", tz_assume_if_naive=None, day_def="trading")
#print_mnq_dataset_info(info_mnq_ret_90_test)


Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...
Archivo encontrado en disco. Cargando dataset local...


# **2. Definición global de tamaños de ventana**

El `window_size` está condicionado por el feature que más historial necesita, en nuestro caso `roc_60`, necesitan 60 minutos previos para poder calcular su primer valor válido.

Si hacemos más corto el window_size corremos el riesgo de perder información o generar NaNs.


## **2.1. Implementación**

In [77]:
# ============================================================
# Definición global de tamaños de ventana (comparabilidad total)
# ============================================================

# Se utilizará la misma grilla de ventanas para:
# - Todos los modelos
# - Todos los horizontes (H=60, H=90)
# - Todos los targets (delta y ret)
# Esto garantiza comparabilidad estricta entre experimentos.
window_sizes = [30, 60, 90, 120, 180]


In [78]:
# ============================================================
# Regla de compatibilidad para ventanas (seq2one / seq2seq)
# ============================================================

def check_window_viability(
    df,
    *,
    window_size: int,
    horizon: int,
    mode: str = "seq2one",  # "seq2one" o "seq2seq"
    date_col: str = "date"
):
    """
    Verifica si un tamaño de ventana es viable para un dataset dado.

    Parámetros
    ----------
    df : DataFrame
        Dataset ya filtrado por split (train/valid/test).
    window_size : int
        Lookback L.
    horizon : int
        Horizonte H.
    mode : str
        "seq2one"  -> requiere al menos L observaciones por día
        "seq2seq"  -> requiere al menos L + H observaciones por día
    """

    if mode not in ["seq2one", "seq2seq"]:
        raise ValueError("mode debe ser 'seq2one' o 'seq2seq'")

    required_length = window_size if mode == "seq2one" else window_size + horizon

    counts_per_day = df.groupby(date_col).size()
    valid_days = counts_per_day[counts_per_day >= required_length]
    invalid_days = counts_per_day[counts_per_day < required_length]

    n_windows_per_day = (
        valid_days - required_length + 1
    ).clip(lower=0)

    n_total_windows = int(n_windows_per_day.sum())

    print(f"Modo: {mode}")
    print(f"L={window_size} | H={horizon}")
    print(f"Días totales        : {len(counts_per_day)}")
    print(f"Días válidos        : {len(valid_days)}")
    print(f"Días descartados    : {len(invalid_days)}")
    print(f"Total ventanas      : {n_total_windows}")

    return {
        "window_size": window_size,
        "horizon": horizon,
        "mode": mode,
        "n_days_total": len(counts_per_day),
        "n_days_valid": len(valid_days),
        "n_days_invalid": len(invalid_days),
        "n_total_windows": n_total_windows,
    }


In [79]:
datasets = {
    "delta_60_train": (mnq_delta_60_train, 60),
    "delta_60_valid": (mnq_delta_60_valid, 60),
    "delta_60_test":  (mnq_delta_60_test,  60),

    "delta_90_train": (mnq_delta_90_train, 90),
    "delta_90_valid": (mnq_delta_90_valid, 90),
    "delta_90_test":  (mnq_delta_90_test,  90),

    "ret_60_train": (mnq_ret_60_train, 60),
    "ret_60_valid": (mnq_ret_60_valid, 60),
    "ret_60_test":  (mnq_ret_60_test,  60),

    "ret_90_train": (mnq_ret_90_train, 90),
    "ret_90_valid": (mnq_ret_90_valid, 90),
    "ret_90_test":  (mnq_ret_90_test,  90),
}

for dataset_name, (df, horizon) in datasets.items():

    print("\n" + "=" * 70)
    print(f"DATASET: {dataset_name} | H={horizon}")
    print("=" * 70)

    for mode in ["seq2one", "seq2seq"]:

        print(f"\n--- MODE: {mode} ---")

        for L in window_sizes:

            check_window_viability(
                df,
                window_size=L,
                horizon=horizon,
                mode=mode
            )


DATASET: delta_60_train | H=60

--- MODE: seq2one ---
Modo: seq2one
L=30 | H=60
Días totales        : 906
Días válidos        : 906
Días descartados    : 0
Total ventanas      : 463872
Modo: seq2one
L=60 | H=60
Días totales        : 906
Días válidos        : 906
Días descartados    : 0
Total ventanas      : 436692
Modo: seq2one
L=90 | H=60
Días totales        : 906
Días válidos        : 906
Días descartados    : 0
Total ventanas      : 409512
Modo: seq2one
L=120 | H=60
Días totales        : 906
Días válidos        : 906
Días descartados    : 0
Total ventanas      : 382332
Modo: seq2one
L=180 | H=60
Días totales        : 906
Días válidos        : 906
Días descartados    : 0
Total ventanas      : 327972

--- MODE: seq2seq ---
Modo: seq2seq
L=30 | H=60
Días totales        : 906
Días válidos        : 906
Días descartados    : 0
Total ventanas      : 409512
Modo: seq2seq
L=60 | H=60
Días totales        : 906
Días válidos        : 906
Días descartados    : 0
Total ventanas      : 382332
Mod

## **2.2. Observaciones**

1. No hay problema estructural

    - 0 días descartados en todos los casos.
    - Todos los `window_sizes` son viables.
    - No existe restricción operativa para usar hasta `L = 180`.

    Desde el punto de vista de los datos, cualquier tamaño de ventana puede utilizarse sin introducir sesgo por pérdida de días.


2. El número de ventanas cae linealmente con L

    Ejemplo (train, H = 60, seq2seq):

    | L   | Ventanas |
    |-----|----------|
    | 30  | 302.784  |
    | 60  | 275.424  |
    | 90  | 248.064  |
    | 120 | 220.704  |
    | 180 | 165.984  |

    Entre `L = 30` y `L = 180` se pierde aproximadamente un 45% de las muestras.

    Esto implica:

    - Mayor varianza para ventanas grandes.
    - Mayor riesgo de overfitting.
    - Mayor costo computacional.


3. Seq2seq penaliza más que seq2one

    Para un mismo L:

    - seq2seq siempre genera menos ventanas.
    - La diferencia aumenta cuando el horizonte H es mayor.

    Ejemplo (H = 90, L = 180):

    - seq2one: 220.704
    - seq2seq: 138.624

    La reducción es significativa, aproximadamente 37%.


4. Conclusión estratégica

    Dado que:

    - No hay restricciones de días.
    - La reducción de ventanas es progresiva y controlada.
    - Se busca comparabilidad estricta entre modelos, targets y horizontes.

    La grilla:

    ```python
    window_sizes = [30, 60, 90, 120, 180]
    ```
    es técnicamente válida y consistente para todo el pipeline.


5. Recomendación profesional

    Para controlar el costo computacional:
    - Primera ronda: evaluar L = 60 y L = 120.
    - Segunda ronda (si es necesario): expandir alrededor del mejor valor.

    Si el objetivo es un análisis académico riguroso y completamente comparable, la grilla completa está correctamente justificada.

6. Resumen final

    - Dataset estructuralmente sólido.
    - Sin pérdidas de días para ningún L.
    - Comparabilidad total entre configuraciones.
    - L = 180 es viable, aunque reduce el tamaño muestral en aproximadamente 40–45%.

    El siguiente paso lógico es evaluar el trade-off entre señal y tamaño muestral durante el entrenamiento.

In [80]:
window_sizes = [30, 60, 90, 120, 180]

# **3. Escalado de los datasets**

## **3.1. Introducción teórica**

El escalado es una etapa **crítica** del pipeline, ya que los modelos de *machine learning* son sensibles a la **escala relativa de las variables de entrada**.  
En este proyecto conviven distintos tipos de variables, entre ellas:

- precios,
- indicadores técnicos,
- interacciones entre indicadores,
- variables temporales,
- y flags binarios,

cada una con **rangos y distribuciones muy diferentes**.

Por este motivo, el escalado debe realizarse de forma **controlada y consistente**, respetando tanto la naturaleza de cada feature como la **coherencia temporal** del dataset.

### **3.1.1. Principios que guían el escalado**


1. **Evitar data leakage**  
   El *scaler* se ajusta (*fit*) **exclusivamente con el conjunto de entrenamiento** y luego se aplica (*transform*) a los conjuntos de validación y test.  
   De este modo se evita introducir información futura durante el entrenamiento.

2. **Escalar solo variables continuas**  
   Se escalan:
   - precios,
   - indicadores técnicos,
   - interacciones,
   - la variable temporal `minute_of_day`.

   No se escalan:
   - flags `_active` (variables binarias),
   - columnas de fecha o identificadores.

3. **Escalar antes de generar las ventanas**  
   El escalado se aplica cuando los datos aún están en formato tabular (`DataFrame`), preservando los nombres de las columnas.  
   Esto permite:
   - seleccionar explícitamente qué columnas se escalan,
   - mantener trazabilidad sobre las features,
   - evitar errores silenciosos una vez que las ventanas se vectorizan y se pierde la referencia a los nombres.

### **3.1.2. Objetivo del escalado**



El objetivo del escalado no es alterar la información contenida en las features, sino **proyectarlas a un espacio numérico comparable**, facilitando que el modelo:

- aprenda relaciones estables entre variables,
- no priorice artificialmente features por su magnitud,
- y generalice correctamente entre distintos días y regímenes intradía.

Con estos criterios establecidos, el siguiente paso consiste en implementar el escalado de forma **explícita y reproducible** para los conjuntos `mnq_train`, `mnq_valid` y `mnq_test`.


## **3.2. Implementación de escalado**

### **3.2.1. Función para elegir escalador**


In [81]:
from __future__ import annotations
from typing import Literal, Optional
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

def choose_scaler(
    scaler_type: str = "standard",
    *,
    with_mean: bool = True,
    with_std: bool = True,
    feature_range: tuple[float, float] = (0.0, 1.0),
    quantile_range: tuple[float, float] = (25.0, 75.0),
) -> Optional[object]:
    """
    Devuelve un scaler de sklearn según `scaler_type`.

    scaler_type soportados:
      - 'standard' | 'z' | 'zscore' -> StandardScaler
      - 'minmax'   | 'min_max'      -> MinMaxScaler
      - 'robust'                   -> RobustScaler
      - 'none' | 'passthrough'     -> None (sin escalado)

    Retorna:
      - instancia de scaler (fit/transform) o None si no se desea escalar.
    """
    st = (scaler_type or "").strip().lower()

    if st in {"standard", "z", "zscore"}:
        return StandardScaler(with_mean=with_mean, with_std=with_std)

    if st in {"minmax", "min_max"}:
        return MinMaxScaler(feature_range=feature_range)

    if st in {"robust"}:
        return RobustScaler(quantile_range=quantile_range, with_centering=True, with_scaling=True)

    if st in {"none", "passthrough"}:
        return None

    raise ValueError(
        f"scaler_type inválido: '{scaler_type}'. "
        "Use: 'standard', 'minmax', 'robust' o 'none'."
    )


### **3.2.2. Función para escalar datasets train, valid y test**


In [82]:
features_to_scale = [
    'minute_of_day', 'close',
    'atr_norm_14', 'atr_norm_20',
    'ema_60',
    'mom_10', 'mom_5',
    'roc_20', 'roc_30', 'roc_60',
    'roc60_x_atr20', 'roc60_x_atr14',
    'roc20_minus_roc60','mom5_minus_mom10',
       ]

In [83]:
import pandas as pd
from typing import Tuple, List, Dict, Any, Sequence

def scale_mnq_splits(
    mnq_train: pd.DataFrame,
    mnq_valid: pd.DataFrame,
    mnq_test: pd.DataFrame,
    *,
    scaler,
    target_cols: str | Sequence[str],
    features_to_scale: Sequence[str] = (
        "minute_of_day", "close",
        "atr_norm_14", "atr_norm_20",
        "ema_60",
        "mom_10", "mom_5",
        "roc_20", "roc_30", "roc_60",
        "roc60_x_atr20", "roc60_x_atr14",
        "roc20_minus_roc60", "mom5_minus_mom10",
    ),
    date_col: str = "date",
    flag_suffix: str = "_flag",
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """
    Escala los splits evitando data leakage.

    Reglas:
      - Fit del scaler SOLO en mnq_train.
      - Escala SOLO las columnas listadas en features_to_scale (si están presentes).
      - NO escala flags (suffix configurable), ni date_col, ni targets.
    """
    # Validación columnas iguales
    cols_train = list(mnq_train.columns)
    if list(mnq_valid.columns) != cols_train or list(mnq_test.columns) != cols_train:
        raise ValueError("mnq_train, mnq_valid y mnq_test deben tener exactamente las mismas columnas y orden.")

    # Normalizar target_cols a lista
    if isinstance(target_cols, str):
        target_cols = [target_cols]
    else:
        target_cols = list(target_cols)

    # Flags y exclusiones
    flag_cols = [c for c in cols_train if c.endswith(flag_suffix)]
    exclude = set([date_col]) | set(flag_cols) | set(target_cols)

    # Escalar SOLO features_to_scale (presentes y no excluidas)
    scale_cols = [c for c in features_to_scale if (c in cols_train and c not in exclude)]

    # Validación mínima
    missing_feats = [c for c in features_to_scale if c not in cols_train]
    if missing_feats:
        # No cortamos: solo informamos en metadata
        pass

    if scaler is None:
        meta = {
            "scaled": False,
            "scale_cols": scale_cols,
            "missing_features_to_scale": missing_feats,
            "flag_cols": flag_cols,
            "target_cols": target_cols,
            "date_col": date_col,
        }
        return mnq_train.copy(), mnq_valid.copy(), mnq_test.copy(), meta

    tr = mnq_train.copy()
    va = mnq_valid.copy()
    te = mnq_test.copy()

    if len(scale_cols) == 0:
        meta = {
            "scaled": False,
            "reason": "No hay columnas para escalar (scale_cols vacío).",
            "scale_cols": scale_cols,
            "missing_features_to_scale": missing_feats,
            "flag_cols": flag_cols,
            "target_cols": target_cols,
            "date_col": date_col,
        }
        return tr, va, te, meta

    # Fit SOLO en train
    scaler.fit(tr[scale_cols])

    # Transform en todos
    tr.loc[:, scale_cols] = scaler.transform(tr[scale_cols])
    va.loc[:, scale_cols] = scaler.transform(va[scale_cols])
    te.loc[:, scale_cols] = scaler.transform(te[scale_cols])

    meta = {
        "scaled": True,
        "scaler_class": scaler.__class__.__name__,
        "scale_cols": scale_cols,
        "missing_features_to_scale": missing_feats,
        "flag_cols": flag_cols,
        "target_cols": target_cols,
        "date_col": date_col,
    }
    return tr, va, te, meta


### **3.2.3. Guardar dataset escalados**


In [84]:
import json
from pathlib import Path
from typing import Dict, Any
import pandas as pd
import joblib


def save_scaled_datasets(
    *,
    mnq_train_scaled: pd.DataFrame,
    mnq_valid_scaled: pd.DataFrame,
    mnq_test_scaled: pd.DataFrame,
    scale_meta: Dict[str, Any],
    out_train_path: Path,
    out_valid_path: Path,
    out_test_path: Path,
    out_meta_path: Path,
    scaler=None,
    out_scaler_path: Path | None = None,
) -> None:
    """
    Guarda datasets escalados, metadata y opcionalmente el scaler.

    No hardcodea rutas.
    Compatible con arquitectura DVC.
    """

    # Crear directorios si no existen
    for path in [out_train_path, out_valid_path, out_test_path, out_meta_path]:
        path.parent.mkdir(parents=True, exist_ok=True)

    # Guardar datasets
    mnq_train_scaled.to_parquet(out_train_path, index=False)
    mnq_valid_scaled.to_parquet(out_valid_path, index=False)
    mnq_test_scaled.to_parquet(out_test_path, index=False)

    # Guardar metadata
    with out_meta_path.open("w", encoding="utf-8") as f:
        json.dump(scale_meta, f, indent=2, ensure_ascii=False)

    # Guardar scaler si se proporciona
    if scaler is not None and out_scaler_path is not None:
        out_scaler_path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(scaler, out_scaler_path)



### **3.2.4. Calcular o cargar datasets escalados**


In [85]:
import json
import joblib
import pandas as pd
from pathlib import Path
from typing import Tuple, Dict, Any, Sequence


def load_or_scale_mnq_datasets(
    *,
    mnq_train: pd.DataFrame,
    mnq_valid: pd.DataFrame,
    mnq_test: pd.DataFrame,
    scaler,
    target_cols: str | Sequence[str],
    out_train_path: Path,
    out_valid_path: Path,
    out_test_path: Path,
    out_meta_path: Path,
    out_scaler_path: Path,
    features_to_scale: Sequence[str] = (
        "minute_of_day", "close",
        "atr_norm_14", "atr_norm_20",
        "ema_60",
        "mom_10", "mom_5",
        "roc_20", "roc_30", "roc_60",
        "roc60_x_atr20", "roc60_x_atr14",
        "roc20_minus_roc60", "mom5_minus_mom10",
    ),
    date_col: str = "date",
    flag_suffix: str = "_flag",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, Any], Any]:
    """
    Carga datasets escalados + scaler si existen.
    Si no existen, escala (fit SOLO en train), guarda datasets, metadata y scaler.

    Retorna:
      mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler
    """

    paths = {
        "train": out_train_path,
        "valid": out_valid_path,
        "test":  out_test_path,
        "meta":  out_meta_path,
        "scaler": out_scaler_path,
    }

    if verbose:
        print("Verificando existencia de datasets escalados y scaler...")
        for k, p in paths.items():
            print(f"  - {k}: {'OK' if p.exists() else 'NO EXISTE'}")

    files_exist = all(p.exists() for p in paths.values())

    # -------------------------------------------------
    # Caso 1: todo existe → cargar y salir
    # -------------------------------------------------
    if files_exist:
        if verbose:
            print("\nTodos los archivos existen. Cargando datasets escalados y scaler...")

        mnq_train_s = pd.read_parquet(out_train_path)
        mnq_valid_s = pd.read_parquet(out_valid_path)
        mnq_test_s  = pd.read_parquet(out_test_path)

        with out_meta_path.open("r", encoding="utf-8") as f:
            scale_meta = json.load(f)

        scaler = joblib.load(out_scaler_path)

        if verbose:
            print("Carga completada. No se recalculó el escalado.")

        return mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler

    # -------------------------------------------------
    # Caso 2: no existe → calcular escalado
    # -------------------------------------------------
    if verbose:
        print("\nNo se encontraron todos los archivos necesarios.")
        print("Recalculando escalado desde cero (fit SOLO en train)...")

    mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta = scale_mnq_splits(
        mnq_train,
        mnq_valid,
        mnq_test,
        scaler=scaler,
        target_cols=target_cols,
        features_to_scale=features_to_scale,
        date_col=date_col,
        flag_suffix=flag_suffix,
    )

    # Crear directorios
    for p in paths.values():
        p.parent.mkdir(parents=True, exist_ok=True)

    # Guardar datasets
    mnq_train_s.to_parquet(out_train_path, index=False)
    mnq_valid_s.to_parquet(out_valid_path, index=False)
    mnq_test_s.to_parquet(out_test_path, index=False)

    # Guardar metadata
    with out_meta_path.open("w", encoding="utf-8") as f:
        json.dump(scale_meta, f, indent=2, ensure_ascii=False)

    # Guardar scaler entrenado (si scaler no es None)
    if scaler is not None:
        joblib.dump(scaler, out_scaler_path)

    if verbose:
        print("Escalado completo y persistido en disco.")

    return mnq_train_s, mnq_valid_s, mnq_test_s, scale_meta, scaler



## **3.3. Aplicación**


In [86]:
from sklearn.preprocessing import StandardScaler

# ============================================================
# DELTA 60
# ============================================================
mnq_delta_60_train_z, mnq_delta_60_valid_z, mnq_delta_60_test_z, meta_delta_60, scaler_delta_60 = load_or_scale_mnq_datasets(
    mnq_train=mnq_delta_60_train,
    mnq_valid=mnq_delta_60_valid,
    mnq_test=mnq_delta_60_test,
    scaler=StandardScaler(),
    target_cols="delta_60",
    out_train_path=OUT_PARQUET_DELTA_60_TRAIN_Z,
    out_valid_path=OUT_PARQUET_DELTA_60_VALID_Z,
    out_test_path=OUT_PARQUET_DELTA_60_TEST_Z,
    out_meta_path=DRIVE_DIR / "data/scaled/meta_delta_60.json",
    out_scaler_path=OUT_SCALER_DELTA_60,
)

# ============================================================
# DELTA 90
# ============================================================
mnq_delta_90_train_z, mnq_delta_90_valid_z, mnq_delta_90_test_z, meta_delta_90, scaler_delta_90 = load_or_scale_mnq_datasets(
    mnq_train=mnq_delta_90_train,
    mnq_valid=mnq_delta_90_valid,
    mnq_test=mnq_delta_90_test,
    scaler=StandardScaler(),
    target_cols="delta_90",
    out_train_path=OUT_PARQUET_DELTA_90_TRAIN_Z,
    out_valid_path=OUT_PARQUET_DELTA_90_VALID_Z,
    out_test_path=OUT_PARQUET_DELTA_90_TEST_Z,
    out_meta_path=DRIVE_DIR / "data/scaled/meta_delta_90.json",
    out_scaler_path=OUT_SCALER_DELTA_90,
)

# ============================================================
# RET 60
# ============================================================
mnq_ret_60_train_z, mnq_ret_60_valid_z, mnq_ret_60_test_z, meta_ret_60, scaler_ret_60 = load_or_scale_mnq_datasets(
    mnq_train=mnq_ret_60_train,
    mnq_valid=mnq_ret_60_valid,
    mnq_test=mnq_ret_60_test,
    scaler=StandardScaler(),
    target_cols="ret_60",
    out_train_path=OUT_PARQUET_RET_60_TRAIN_Z,
    out_valid_path=OUT_PARQUET_RET_60_VALID_Z,
    out_test_path=OUT_PARQUET_RET_60_TEST_Z,
    out_meta_path=DRIVE_DIR / "data/scaled/meta_ret_60.json",
    out_scaler_path=OUT_SCALER_RET_60,
)

# ============================================================
# RET 90
# ============================================================
mnq_ret_90_train_z, mnq_ret_90_valid_z, mnq_ret_90_test_z, meta_ret_90, scaler_ret_90 = load_or_scale_mnq_datasets(
    mnq_train=mnq_ret_90_train,
    mnq_valid=mnq_ret_90_valid,
    mnq_test=mnq_ret_90_test,
    scaler=StandardScaler(),
    target_cols="ret_90",
    out_train_path=OUT_PARQUET_RET_90_TRAIN_Z,
    out_valid_path=OUT_PARQUET_RET_90_VALID_Z,
    out_test_path=OUT_PARQUET_RET_90_TEST_Z,
    out_meta_path=DRIVE_DIR / "data/scaled/meta_ret_90.json",
    out_scaler_path=OUT_SCALER_RET_90,
)


Verificando existencia de datasets escalados y scaler...
  - train: OK
  - valid: OK
  - test: OK
  - meta: OK
  - scaler: OK

Todos los archivos existen. Cargando datasets escalados y scaler...
Carga completada. No se recalculó el escalado.
Verificando existencia de datasets escalados y scaler...
  - train: OK
  - valid: OK
  - test: OK
  - meta: OK
  - scaler: OK

Todos los archivos existen. Cargando datasets escalados y scaler...
Carga completada. No se recalculó el escalado.
Verificando existencia de datasets escalados y scaler...
  - train: OK
  - valid: OK
  - test: OK
  - meta: OK
  - scaler: OK

Todos los archivos existen. Cargando datasets escalados y scaler...
Carga completada. No se recalculó el escalado.
Verificando existencia de datasets escalados y scaler...
  - train: OK
  - valid: OK
  - test: OK
  - meta: OK
  - scaler: OK

Todos los archivos existen. Cargando datasets escalados y scaler...
Carga completada. No se recalculó el escalado.


## **3.4. Verificación**


In [87]:
import pandas as pd

def verify_scaling(
    train_raw: pd.DataFrame,
    train_z: pd.DataFrame,
    valid_raw: pd.DataFrame,
    valid_z: pd.DataFrame,
    test_raw: pd.DataFrame,
    test_z: pd.DataFrame,
    *,
    name: str,
    target_col: str,
    features_to_scale: list[str],
    date_col: str = "date",
    tol_mean: float = 0.05,
    tol_std: float = 0.05,
    show_bad_tables: bool = True,
) -> None:
    """
    Verifica escalado de forma completa:

    1) TRAIN: mean≈0 y std≈1 para features_to_scale presentes.
    2) VALID/TEST: confirma que fueron transformados (no iguales al raw).
    3) Target y date: confirma que NO fueron modificados.
    """
    print(f"\n==================== {name} ====================")

    cols = [c for c in features_to_scale if c in train_z.columns]
    print(f"Features a verificar (presentes): {len(cols)}")

    if len(cols) == 0:
        print("No hay columnas de features_to_scale presentes para verificar.")
        return

    # 1) TRAIN stats
    stats = train_z[cols].agg(["mean", "std"]).T
    stats["mean_abs"] = stats["mean"].abs()
    stats["std_err"] = (stats["std"] - 1.0).abs()

    bad_mean = stats[stats["mean_abs"] > tol_mean]
    bad_std  = stats[stats["std_err"] > tol_std]

    print("\nTRAIN (estandarización):")
    print(f"  Mean fuera tolerancia (|mean| > {tol_mean}): {len(bad_mean)}")
    print(f"  Std  fuera tolerancia (|std-1| > {tol_std}): {len(bad_std)}")

    if show_bad_tables:
        if len(bad_mean) > 0:
            print("\nColumnas con mean fuera de tolerancia:")
            display(bad_mean[["mean", "std"]].sort_values("mean", key=lambda s: s.abs(), ascending=False))
        if len(bad_std) > 0:
            print("\nColumnas con std fuera de tolerancia:")
            display(bad_std[["mean", "std"]].sort_values("std_err", ascending=False))

    # 2) VALID/TEST transformados (solo sobre cols)
    valid_changed = not valid_raw[cols].equals(valid_z[cols])
    test_changed  = not test_raw[cols].equals(test_z[cols])

    print("\nTransformación aplicada:")
    print(f"  VALID transformado: {valid_changed}")
    print(f"  TEST  transformado: {test_changed}")

    # 3) Target sin cambios
    if target_col in train_raw.columns and target_col in train_z.columns:
        same_target_train = train_raw[target_col].equals(train_z[target_col])
        same_target_valid = valid_raw[target_col].equals(valid_z[target_col]) if target_col in valid_raw.columns and target_col in valid_z.columns else None
        same_target_test  = test_raw[target_col].equals(test_z[target_col])   if target_col in test_raw.columns and target_col in test_z.columns else None

        print("\nTarget sin modificar:")
        print(f"  TRAIN: {same_target_train}")
        if same_target_valid is not None:
            print(f"  VALID: {same_target_valid}")
        if same_target_test is not None:
            print(f"  TEST : {same_target_test}")
    else:
        print(f"\nTarget '{target_col}' no encontrado en raw/z para comparar.")

    # date sin cambios
    if date_col in train_raw.columns and date_col in train_z.columns:
        same_date_train = train_raw[date_col].equals(train_z[date_col])
        same_date_valid = valid_raw[date_col].equals(valid_z[date_col]) if date_col in valid_raw.columns and date_col in valid_z.columns else None
        same_date_test  = test_raw[date_col].equals(test_z[date_col])   if date_col in test_raw.columns and date_col in test_z.columns else None

        print("\nDate sin modificar:")
        print(f"  TRAIN: {same_date_train}")
        if same_date_valid is not None:
            print(f"  VALID: {same_date_valid}")
        if same_date_test is not None:
            print(f"  TEST : {same_date_test}")



In [88]:
# =========================
# DELTA 60
# =========================
verify_scaling(
    mnq_delta_60_train, mnq_delta_60_train_z,
    mnq_delta_60_valid, mnq_delta_60_valid_z,
    mnq_delta_60_test,  mnq_delta_60_test_z,
    name="delta_60",
    target_col="delta_60",
    features_to_scale=features_to_scale,
)

# =========================
# DELTA 90
# =========================
verify_scaling(
    mnq_delta_90_train, mnq_delta_90_train_z,
    mnq_delta_90_valid, mnq_delta_90_valid_z,
    mnq_delta_90_test,  mnq_delta_90_test_z,
    name="delta_90",
    target_col="delta_90",
    features_to_scale=features_to_scale,
)

# =========================
# RET 60
# =========================
verify_scaling(
    mnq_ret_60_train, mnq_ret_60_train_z,
    mnq_ret_60_valid, mnq_ret_60_valid_z,
    mnq_ret_60_test,  mnq_ret_60_test_z,
    name="ret_60",
    target_col="ret_60",
    features_to_scale=features_to_scale,
)

# =========================
# RET 90
# =========================
verify_scaling(
    mnq_ret_90_train, mnq_ret_90_train_z,
    mnq_ret_90_valid, mnq_ret_90_valid_z,
    mnq_ret_90_test,  mnq_ret_90_test_z,
    name="ret_90",
    target_col="ret_90",
    features_to_scale=features_to_scale,
)



==================== delta_60 ====================
Features a verificar (presentes): 14

TRAIN (estandarización):
  Mean fuera tolerancia (|mean| > 0.05): 0
  Std  fuera tolerancia (|std-1| > 0.05): 0

Transformación aplicada:
  VALID transformado: True
  TEST  transformado: True

Target sin modificar:
  TRAIN: False
  VALID: False
  TEST : False

Date sin modificar:
  TRAIN: False
  VALID: False
  TEST : False

==================== delta_90 ====================
Features a verificar (presentes): 14

TRAIN (estandarización):
  Mean fuera tolerancia (|mean| > 0.05): 0
  Std  fuera tolerancia (|std-1| > 0.05): 0

Transformación aplicada:
  VALID transformado: True
  TEST  transformado: True

Target sin modificar:
  TRAIN: False
  VALID: False
  TEST : False

Date sin modificar:
  TRAIN: False
  VALID: False
  TEST : False

==================== ret_60 ====================
Features a verificar (presentes): 14

TRAIN (estandarización):
  Mean fuera tolerancia (|mean| > 0.05): 0
  Std  fuera

# **4. Generación de ventanas deslizantes (sliding windows)**

## **4.1. Introducción conceptual**

**1. Objetivo de esta etapa**

En esta etapa se generan datasets de entrenamiento para modelos intradía a partir de los splits `train/valid/test`, construyendo ventanas temporales consecutivas (sliding windows) dentro de cada día de operación. Se construirán dos variantes:

- Ventanas seq2one: una secuencia de entrada produce un único valor objetivo.
- Ventanas seq2seq: una secuencia de entrada produce una secuencia objetivo alineada en el tiempo.

La generación se realiza para múltiples tamaños de ventana, definidos por una lista única:

```python
window_sizes = [30, 60, 90, 120, 180]
```

Este enfoque permite comparar modelos manteniendo la misma grilla de lookback para todos los targets (delta/ret) y horizontes (60/90).

**2. Organización temporal y prevención de leakage**

- El dataset está organizado por días de operación (`date`).
- Los splits `train/valid/test` se realizan a nivel diario: no existen días compartidos entre splits.
- Las ventanas se generan de forma independiente dentro de cada día, sin cruzar el límite diario.
- El orden temporal se preserva estrictamente: las ventanas se generan en forma consecutiva, no aleatoria.

**3. Ventanas deslizantes dentro de cada día**

Para cada día con N registros minuto a minuto, se generan ventanas que comienzan en el índice `i` y terminan en:

- Fin de ventana: `i + window_size - 1`

La cantidad de ventanas por día depende del tipo de esquema:

- Seq2one: `N - window_size + 1`
- Seq2seq (horizonte H): `N - (window_size + H) + 1`

Esto asegura que el target se toma siempre de información futura respecto de la secuencia de entrada, manteniendo coherencia causal.

**4. Esquema seq2one (many-to-one)**

- Definición

  - Entrada: una secuencia de `window_size` minutos con F features por minuto.
  - Salida: un único valor objetivo (escalar).

- Construcción

  Para cada día, se recorre `i = 0, 1, 2, ...` y se define:

  - `X_i`: registros `[i : i + window_size]` (longitud `window_size`)
  - `y_i`: target del último registro de la ventana, es decir el target en el tiempo `i + window_size - 1`

  Ejemplo conceptual (N=421, window_size=60)

  | Iteración | Ventana usada     | Target extraído       |
  |------------|------------------|------------------------|
  | i = 0      | registros 0-59   | target = registro 59   |
  | i = 1      | registros 1-60   | target = registro 60   |
  | ...        | ...              | ...                    |
  | i = 361    | registros 361-420| target = registro 420  |

- Ventanas por día:

    `421 - 60 + 1 = 362`

- Dimensiones

  - Entrada: `window_size × F`
  - Salida: `1`

**5. Esquema seq2seq (many-to-many)**

- Definición

  - Entrada: una secuencia de `window_size` minutos con F features por minuto.
  - Salida: una secuencia de `window_size` targets, alineados minuto a minuto con la entrada.


- Construcción

  Para cada día, se recorre `i = 0, 1, 2, ...` y se define:

  - `X_i`: registros `[i : i + window_size]`
  - `Y_i`: targets alineados `[i : i + window_size]`


- Interpretación
  - El modelo consume una secuencia de `window_size` minutos.
  - El objetivo es predecir el vector completo de targets asociados a esos mismos minutos (alineación 1 a 1 entre filas de `X_i` y posiciones de `Y_i`).
  - El horizonte `H` no define la longitud de la salida: ya está incorporado en el target (por ejemplo `delta_60`, `delta_90`, `ret_60`, `ret_90`).

- Dimensiones
  - Entrada: `window_size x F`
  - Salida: `window_size` (o `window_size x 1` según representación)

**6. Compatibilidad con targets y horizontes**

  En el proyecto existen cuatro combinaciones base (target / horizonte implícito):

  - delta_60
  - delta_90
  - ret_60
  - ret_90

  Cada uno de estos targets ya incorpora internamente el horizonte hacia adelante (60 o 90 minutos). Por lo tanto, el horizonte no se modela explícitamente durante la generación de ventanas, sino que está embebido en la definición del target.

  Se generarán ventanas para cada uno de estos targets reutilizando la misma lista:

  `window_sizes = [30, 60, 90, 120, 180]`

  Notas

  - En seq2one:
    - La salida siempre es un escalar.
    - El horizonte no afecta la dimensión de `y`, únicamente modifica el significado económico del target (por ejemplo delta_60 vs delta_90).

  - En seq2seq alineado:
    - La salida tiene la misma longitud que `window_size`.
    - El horizonte tampoco afecta la dimensión de `Y`.
    - La cantidad de ventanas por día depende únicamente de `window_size`, no del horizonte.


**7. Propiedades garantizadas del esquema**

  - No hay mezcla de días entre splits.
  - No hay cruces entre días al construir ventanas.
  - Las ventanas respetan estrictamente el orden temporal.
  - No se introduce leakage: el scaler se ajusta solo con train y las ventanas se construyen dentro de cada split.
  - La comparación entre modelos es consistente porque:
    - se usa la misma grilla `window_sizes`,
    - los targets/horizontes se tratan de manera homogénea,
    - y se preserva el mismo protocolo temporal.






## **4.2. Construcción de ventanas `seq2one`**

### **4.2.1. Generador seq2one**

In [46]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from typing import List, Tuple

def generate_windows_seq2one(
    df: pd.DataFrame,
    *,
    date_col: str,
    features: List[str],
    target_col: str,
    window_size: int,
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Genera ventanas "seq2one" (many-to-one) a partir de un DataFrame intradía.

    Idea general
    ------------
    - Se asume que el DataFrame contiene múltiples días (o grupos) identificados por `date_col`.
    - Para cada día, se construyen ventanas deslizantes (sliding windows) de longitud `window_size`
      sobre las columnas `features`.
    - La etiqueta (target) asociada a cada ventana es un único valor (one) tomado del vector `target_col`
      alineado al final de la ventana: yw[t] = yg[t + window_size - 1].
      (Es decir, la ventana que termina en el índice i predice el target en ese índice i).

    Ahorro de espacio
    -----------------
    - X se devuelve en float16 para reducir tamaño (aprox. la mitad vs float32).
    - y se mantiene en float32 para preservar estabilidad numérica en el target y métricas.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con columnas [date_col] + features + [target_col].
    date_col : str
        Columna que define el agrupamiento por día/sesión (ej: 'date').
    features : List[str]
        Lista de columnas numéricas usadas como entradas.
    target_col : str
        Columna numérica objetivo.
    window_size : int
        Longitud L de la ventana.
    flatten : bool
        Si True, retorna X aplanado a 2D: (N, L*F) en lugar de (N, L, F).
        Útil para modelos tipo MLP/Ridge, etc.
    drop_windows_with_nan : bool
        Si True, descarta ventanas donde X o y contengan NaN.

    Retorna
    -------
    X : np.ndarray
        - Si flatten=False: shape (N, L, F), dtype float16
        - Si flatten=True : shape (N, L*F), dtype float16
    y : np.ndarray
        shape (N,), dtype float32
    """
    # -------------------------
    # Validaciones básicas
    # -------------------------
    if window_size <= 0:
        raise ValueError("window_size debe ser un entero positivo.")

    # Verificar que existan todas las columnas necesarias en df
    required_cols = [date_col] + features + [target_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en df: {missing}")

    # Acumuladores para juntar ventanas de todos los días
    X_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []

    # Cantidad de features (F) para validar shapes
    F = len(features)

    # -------------------------
    # Construcción por día
    # -------------------------
    # Importante: se agrupa por date_col para evitar "leakage" entre días
    # (no se construyen ventanas que crucen el límite de un día/sesión).
    for _, g in df.groupby(date_col, sort=False):
        # Reset index para tener índices [0..n-1] por cada grupo
        g = g.reset_index(drop=True)
        n = len(g)

        # Si el día tiene menos filas que la ventana, no puede generar ventanas
        if n < window_size:
            continue

        # -------------------------
        # Conversión a numpy
        # -------------------------
        # Xg: (n, F) en float32 para estabilidad numérica durante el armado
        # Nota: la reducción a float16 la hacemos al final sobre Xw (ventanas ya formadas)
        Xg = g[features].to_numpy(dtype=np.float32, copy=False)

        # yg: (n,) en float32 (target)
        yg = g[target_col].to_numpy(dtype=np.float32, copy=False)

        # -------------------------
        # Ventanas deslizantes
        # -------------------------
        # sliding_window_view crea una vista (no copia) con ventanas sobre el eje 0.
        # Idealmente devuelve (n-L+1, L, F), pero según la versión/forma puede venir como
        # (n-L+1, F, L). Por eso normalizamos luego.
        Xw = sliding_window_view(Xg, window_shape=window_size, axis=0)

        # Verificación de dimensionalidad esperada: debe ser 3D
        if Xw.ndim != 3:
            raise ValueError(f"Xw ndim inesperado: {Xw.ndim} | shape={Xw.shape}")

        # -------------------------
        # Normalización de ejes (compatibilidad numpy)
        # -------------------------
        # Caso observado: (N, F, L) en lugar de (N, L, F)
        # Si detectamos ese patrón, swapaxes(1,2) lo corrige.
        if Xw.shape[1] == F and Xw.shape[2] == window_size:
            Xw = np.swapaxes(Xw, 1, 2)

        # Verificación final de shape: (N, L, F)
        if not (Xw.shape[1] == window_size and Xw.shape[2] == F):
            raise ValueError(
                f"Xw shape inválido tras normalizar: {Xw.shape} "
                f"(esperado: (N,{window_size},{F}))"
            )

        # -------------------------
        # Construcción del target por ventana (seq2one)
        # -------------------------
        # Cada ventana de Xw "termina" en el índice i (del día),
        # y su etiqueta será el yg en ese mismo índice i.
        # Por eso se recorta yg desde (window_size-1) hasta el final.
        yw = yg[window_size - 1 :].astype(np.float32, copy=False)

        # -------------------------
        # Filtrado de NaNs (opcional)
        # -------------------------
        if drop_windows_with_nan:
            # Ventana válida si NO hay NaN en ningún elemento de (L, F)
            x_ok = ~np.isnan(Xw).any(axis=(1, 2))

            # y válido si no es NaN
            # (yw es np.ndarray, pero usamos pd.isna por compatibilidad general)
            y_ok = ~pd.isna(yw)

            # Mantener solo índices válidos en ambos
            ok = x_ok & y_ok
            Xw = Xw[ok]
            yw = yw[ok]

        # Si luego del filtro no queda nada, saltar el día
        if Xw.size == 0:
            continue

        # -------------------------
        # Aplanado (opcional)
        # -------------------------
        # Para modelos 2D (MLP/Ridge/etc.), se suele aplanar (N, L, F) -> (N, L*F)
        if flatten:
            Xw = Xw.reshape(Xw.shape[0], -1)

        # -------------------------
        # Cast a float16 solo para X
        # -------------------------
        # Esto reduce el tamaño en disco y RAM aproximadamente 50% vs float32.
        # y se mantiene en float32 para no degradar el target ni la evaluación.
        X_all.append(Xw.astype(np.float16, copy=False))
        y_all.append(yw)

    # -------------------------
    # Caso borde: sin ventanas
    # -------------------------
    if not X_all:
        # Retornar arrays vacíos con shapes consistentes
        X = np.empty((0, window_size, F), dtype=np.float16)
        if flatten:
            X = X.reshape(0, window_size * F)
        y = np.empty((0,), dtype=np.float32)
        return X, y

    # -------------------------
    # Concatenación final (todos los días)
    # -------------------------
    # X: (N_total, L, F) o (N_total, L*F)
    # y: (N_total,)
    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)

    return X, y

### **4.2.2. Load/Build para SEQ2ONE**

In [47]:
import numpy as np
from pathlib import Path
from typing import List, Tuple

def prepare_or_load_seq2one_windows_npz(
    *,
    mnq_train,
    mnq_valid,
    mnq_test,
    features: List[str],
    target_col: str,
    window_size: int,

    # Se pasa 1 path por split: un .npz que contiene dos arrays (X e y)
    out_train,
    out_valid,
    out_test,

    date_col: str = "date",
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
    verbose: bool = True,
    repair_axis_order_if_needed: bool = True,

    # Nota: en .npz no hay un mmap real como en .npy; se deja por compatibilidad
    mmap_mode: str | None = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Prepara o carga ventanas seq2one desde disco en formato .npz (un archivo por split).

    Flujo
    -----
    Para cada split (train/valid/test):
      - Si existe el archivo .npz: lo carga (X, y).
      - Si NO existe: construye ventanas con generate_windows_seq2one(...) y las guarda en el .npz.
      - Opcional: repara el orden de ejes de X si detecta (N, F, L) en lugar de (N, L, F).

    Entradas
    --------
    - mnq_train/mnq_valid/mnq_test: DataFrames o estructuras compatibles con generate_windows_seq2one.
    - features, target_col, window_size: definen cómo construir las ventanas.
    - out_train/out_valid/out_test: rutas destino para los .npz.

    Retorna
    -------
    (X_train, y_train, X_valid, y_valid, X_test, y_test)
    """

    # -------------------------
    # Helpers de path
    # -------------------------
    def _to_path(p) -> Path:
        # Permite que out_* sea Path o string (u otro objeto convertible a string)
        return p if isinstance(p, Path) else Path(str(p))

    def _ensure_parent_dir(path: Path) -> None:
        # Crea el directorio padre si no existe (DVC/Colab friendly)
        path.parent.mkdir(parents=True, exist_ok=True)

    # Cantidad de features, usada para validaciones de shape y reparación de ejes
    F = len(features)

    # -------------------------
    # Reparación opcional del orden de ejes en X
    # -------------------------
    def _maybe_repair_X(X: np.ndarray) -> np.ndarray:
        """
        Algunos flujos pueden guardar X como (N, F, L) en vez de (N, L, F).
        Este helper detecta ese patrón y lo corrige con swapaxes.

        Se aplica SOLO si:
        - flatten=False (porque si X es 2D, no corresponde)
        - repair_axis_order_if_needed=True
        """
        if flatten or not repair_axis_order_if_needed:
            return X

        # Si X tiene 3 dimensiones y coincide (N, F, L), lo convertimos a (N, L, F)
        if X.ndim == 3 and X.shape[1] == F and X.shape[2] == window_size:
            return np.swapaxes(X, 1, 2)  # (N,F,L)->(N,L,F)

        return X

    # -------------------------
    # Guardado y carga NPZ
    # -------------------------
    def _save_npz(path_npz: Path, X: np.ndarray, y: np.ndarray) -> None:
        _ensure_parent_dir(path_npz)

        # Forzar dtype antes de guardar (garantía de consistencia)
        X = np.asarray(X, dtype=np.float16)
        y = np.asarray(y, dtype=np.float32)

        np.savez_compressed(path_npz, X=X, y=y)

    def _load_npz(path_npz: Path):
        """
        Carga X e y desde un .npz.
        - allow_pickle=False por seguridad y consistencia.
        - mmap_mode no aplica realmente en .npz; se mantiene solo como parámetro de compatibilidad.
        """
        with np.load(path_npz, allow_pickle=False) as z:
            X = z["X"]
            y = z["y"]
        return X, y

    # -------------------------
    # Lógica principal por split
    # -------------------------
    def _load_or_build(split_name: str, df, path_npz):
        """
        Para un split:
        - Si existe path_npz: LOAD
        - Si no existe: BUILD (genera ventanas, guarda)
        - Luego: opcionalmente repara ejes y re-guarda si estaba mal
        """
        path_npz = _to_path(path_npz)

        action = "load"
        if not path_npz.exists():
            # Si el archivo no existe, construimos ventanas desde el df y guardamos
            action = "build"
            X, y = generate_windows_seq2one(
                df=df,
                date_col=date_col,
                features=features,
                target_col=target_col,
                window_size=window_size,
                flatten=flatten,
                drop_windows_with_nan=drop_windows_with_nan,
            )
            _save_npz(path_npz, X, y)
        else:
            # Si existe, simplemente cargamos
            X, y = _load_npz(path_npz)

        # -------------------------
        # Reparación opcional de X si estaba guardado como (N,F,L)
        # -------------------------
        X2 = _maybe_repair_X(X)

        # Si realmente cambió la vista/orden de ejes y venía de LOAD,
        # conviene persistir la corrección re-guardando el .npz
        if (X2 is not X) and action == "load":
            X2_np = np.asarray(X2)  # materializa (por si fuera vista)
            y_np = np.asarray(y)
            _save_npz(path_npz, X2_np, y_np)
            X = X2_np
            y = y_np
        else:
            X = X2

        # Log simple para trazabilidad
        if verbose:
            print(
                f"[{target_col} | L={window_size} | {split_name}] {action.upper()}  "
                f"X={X.shape}  y={y.shape}  -> {path_npz.name}"
            )

        return X, y

    # Ejecutar para cada split
    X_train, y_train = _load_or_build("train", mnq_train, out_train)
    X_valid, y_valid = _load_or_build("valid", mnq_valid, out_valid)
    X_test,  y_test  = _load_or_build("test",  mnq_test,  out_test)

    return X_train, y_train, X_valid, y_valid, X_test, y_test

### **4.2.3. Info para SEQ2ONE**

In [48]:
from pathlib import Path
import numpy as np

def xy_info_seq2one_compact(
    target_col,
    window_size_name,
    horizon_min: int,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    *,
    window_size: int,
    n_features: int,
    # Opcional: rutas a los .npz (para mostrar formato y tamaño)
    path_train: str | Path | None = None,
    path_valid: str | Path | None = None,
    path_test:  str | Path | None = None,
):
    """
    Muestra información estructural compacta por split, incluyendo:
    - shape y validaciones de dimensiones
    - dtype de X e y
    - (opcional) tamaño de archivo y estimación de "npz-compressed" vs "npz"
      * Nota: la compresión real no se puede afirmar al 100% solo con X/y en RAM.
        Si se provee el path, se puede estimar comparando tamaño en disco vs tamaño bruto.
    """

    def _to_path(p):
        return None if p is None else (p if isinstance(p, Path) else Path(str(p)))

    def _human_bytes(n: int) -> str:
        units = ["B", "KB", "MB", "GB", "TB"]
        x = float(n)
        for u in units:
            if x < 1024.0 or u == units[-1]:
                return f"{x:.2f}{u}"
            x /= 1024.0
        return f"{x:.2f}TB"

    def _file_info(path: Path | None, X: np.ndarray, y: np.ndarray) -> str:
        # Si no hay path o el archivo no existe, no se puede informar tamaño/formato
        if path is None or not path.exists():
            return "file=NA"

        size_disk = path.stat().st_size

        # Tamaño "bruto" aproximado si se guardara sin compresión:
        # np.savez almacena cada array como .npy dentro del zip, que incluye header,
        # aquí aproximamos con nbytes (sin headers).
        raw_est = int(getattr(X, "nbytes", 0) + getattr(y, "nbytes", 0))

        # Ratio para estimar si hubo compresión efectiva
        # (si el ratio es significativamente menor a 1, es muy probable compressed)
        ratio = (size_disk / raw_est) if raw_est > 0 else float("nan")

        # Heurística simple
        if raw_est > 0 and ratio < 0.85:
            kind = "npz-compressed (probable)"
        else:
            kind = "npz (sin compresión o compresión mínima)"

        return f"file={_human_bytes(size_disk)} | raw~{_human_bytes(raw_est)} | {kind} | ratio={ratio:.3f}"

    p_train = _to_path(path_train)
    p_valid = _to_path(path_valid)
    p_test  = _to_path(path_test)

    def _print_line(name, X, y, path: Path | None):
        if not hasattr(X, "shape") or not hasattr(y, "shape"):
            print(f"{name} | ERROR: X o y sin shape")
            return

        N = X.shape[0] if X.ndim >= 1 else 0

        # Validación de dimensiones esperadas
        if X.ndim == 3:
            L, F = X.shape[1], X.shape[2]
            ok = (L == window_size) and (F == n_features)
            dim_info = f"(L,F)=({L},{F})"
        elif X.ndim == 2:
            dim = X.shape[1]
            expected = window_size * n_features
            ok = (dim == expected)
            dim_info = f"dim={dim}"
        else:
            ok = False
            dim_info = "dim=?"

        y_ok = (y.ndim == 1 and y.shape[0] == N) or (y.ndim == 2 and y.shape == (N, 1))
        status = "OK" if ok and y_ok else "ERROR"

        # dtype info
        x_dtype = getattr(X, "dtype", "NA")
        y_dtype = getattr(y, "dtype", "NA")

        # file info (si hay path)
        finfo = _file_info(path, X, y)

        print(
            f"{name:<35} | "
            f"X:{X.shape} ({x_dtype}) | "
            f"y:{y.shape} ({y_dtype}) | "
            f"N={N} | "
            f"{dim_info} | "
            f"{status} | "
            f"{finfo}"
        )

    _print_line(f"[{target_col} | L={window_size} | train]", X_train, y_train, p_train)
    _print_line(f"[{target_col} | L={window_size} | valid]", X_valid, y_valid, p_valid)
    _print_line(f"[{target_col} | L={window_size} | test ]", X_test,  y_test,  p_test)
    print()

In [49]:
INFO_SEQ2ONE_WINDOWS = '''
xy_info_seq2one_compact(
    "delta_60", "L60", 60,
    X_train, y_train, X_valid, y_valid, X_test, y_test,
    window_size=60, n_features=36,
    path_train=OUT_SEQ2ONE_DELTA_60_TRAIN,
    path_valid=OUT_SEQ2ONE_DELTA_60_VALID,
    path_test=OUT_SEQ2ONE_DELTA_60_TEST,
)
'''

### **4.2.4. Utilidades**

In [50]:
features_to_windows = [
    'minute_of_day',
    'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_overnight',
    'is_mon', 'is_tue', 'is_wed', 'is_thu','is_fri',
    'close',
    'atr_norm_14', 'atr_norm_14_flag', 'atr_norm_20', 'atr_norm_20_flag',
    'ema_60', 'ema_60_flag',
    'mom_10', 'mom_10_flag', 'mom_5', 'mom_5_flag',
    'roc_20', 'roc_20_flag', 'roc_30', 'roc_30_flag', 'roc_60', 'roc_60_flag',
    'roc60_x_atr20', 'roc60_x_atr20_flag',
    'roc60_x_atr14', 'roc60_x_atr14_flag',
    'roc20_minus_roc60', 'roc20_minus_roc60_flag',
    'mom5_minus_mom10', 'mom5_minus_mom10_flag',
    ]

In [51]:
n_features = len(features_to_windows)

In [52]:
# Mapa target -> datasets escalados
targets_windows = {
    "delta_60": (mnq_delta_60_train_z, mnq_delta_60_valid_z, mnq_delta_60_test_z),
    "delta_90": (mnq_delta_90_train_z, mnq_delta_90_valid_z, mnq_delta_90_test_z),
    "ret_60":   (mnq_ret_60_train_z,   mnq_ret_60_valid_z,   mnq_ret_60_test_z),
    "ret_90":   (mnq_ret_90_train_z,   mnq_ret_90_valid_z,   mnq_ret_90_test_z),
}

In [53]:
OUT_WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / "data/windows/seq2one"

### **4.2.5. Cargar ventanas .npy**

In [54]:
def _load_npz(path_npz: Path):
        """
        Carga X e y desde un .npz.
        - allow_pickle=False por seguridad y consistencia.
        - mmap_mode no aplica realmente en .npz; se mantiene solo como parámetro de compatibilidad.
        """
        with np.load(path_npz, allow_pickle=False) as z:
            X = z["X"]
            y = z["y"]
        return X, y

### **4.2.6. Generación de resumen de ventanas `seq2one`**

In [57]:
#1) Helpers: convertir bytes y extraer info de cada split
import json
from pathlib import Path
import numpy as np
from datetime import datetime

def _human_bytes(n: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)
    for u in units:
        if x < 1024.0 or u == units[-1]:
            return f"{x:.2f}{u}"
        x /= 1024.0
    return f"{x:.2f}TB"

def _split_file_info(path: Path, X: np.ndarray, y: np.ndarray) -> dict:
    """
    Devuelve info del archivo + shapes/dtypes.
    - size_disk: tamaño real del .npz
    - raw_est: tamaño bruto aproximado (X.nbytes + y.nbytes)
    - ratio: size_disk/raw_est (si < ~0.85 sugiere compresión efectiva)
    """
    path = Path(path)
    size_disk = path.stat().st_size if path.exists() else None
    raw_est = int(getattr(X, "nbytes", 0) + getattr(y, "nbytes", 0))
    ratio = (size_disk / raw_est) if (size_disk is not None and raw_est > 0) else None

    kind = None
    if ratio is not None:
        kind = "npz-compressed (probable)" if ratio < 0.85 else "npz (sin compresión o mínima)"

    return {
        "file_name": path.name,
        "file_path": str(path),
        "size_disk_bytes": size_disk,
        "size_disk_human": _human_bytes(size_disk) if size_disk is not None else None,
        "raw_estimated_bytes": raw_est,
        "raw_estimated_human": _human_bytes(raw_est),
        "compression_ratio": round(ratio, 6) if ratio is not None else None,
        "compression_kind": kind,
        "shape_X": list(X.shape),
        "shape_y": list(y.shape),
        "dtype_X": str(getattr(X, "dtype", "NA")),
        "dtype_y": str(getattr(y, "dtype", "NA")),
    }

In [58]:
#2) Builder del manifest (se llama en cada iteración)
def update_seq2one_manifest(
    manifest: dict,
    *,
    target_col: str,
    window_size: int,
    n_features: int,
    features: list[str],
    X_train, y_train, X_valid, y_valid, X_test, y_test,
    path_train: Path,
    path_valid: Path,
    path_test: Path,
    date_col: str = "date",
    flatten: bool = False,
) -> dict:
    """
    Actualiza el manifest global con la entrada (target_col, window_size).
    Retorna el manifest actualizado (mutación in-place también).
    """
    key = f"{target_col}__L{window_size}"

    manifest["items"][key] = {
        "target": target_col,
        "window_size": int(window_size),
        "n_features": int(n_features),
        "features": list(features),
        "date_col": date_col,
        "flatten": bool(flatten),
        "format_expected": "npz_compressed",
        "splits": {
            "train": _split_file_info(Path(path_train), np.asarray(X_train), np.asarray(y_train)),
            "valid": _split_file_info(Path(path_valid), np.asarray(X_valid), np.asarray(y_valid)),
            "test":  _split_file_info(Path(path_test),  np.asarray(X_test),  np.asarray(y_test)),
        },
    }

    return manifest

In [59]:
#3) Guardar el JSON global al final
def save_seq2one_manifest(manifest: dict, *, out_path: Path) -> Path:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    print(f"[MANIFEST] Guardado: {out_path}")
    return out_path

### **4.2.7. Generación de ventanas `seq2one`**

In [60]:
from pathlib import Path
import numpy as np

manifest = {
    "schema": "seq2one_windows_manifest_v1",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "items": {}
}

for target_col, (df_tr, df_va, df_te) in targets_windows.items():
    for L in window_sizes:

        print("\n" + "=" * 70)
        print(f"SEQ2ONE | TARGET={target_col} | L={L} | F={n_features}")
        print("=" * 70)

        # Directorio por window size (L)
        out_dir_L = OUT_WINDOWS_SEQ2ONE_DIR / f"L{L}"
        out_dir_L.mkdir(parents=True, exist_ok=True)

        # Un único archivo NPZ por split (contiene X e y con claves 'X' y 'y')
        out_train = out_dir_L / f"windows_{target_col}_train.npz"
        out_valid = out_dir_L / f"windows_{target_col}_valid.npz"
        out_test  = out_dir_L / f"windows_{target_col}_test.npz"

        # Verificamos si ya existen los 3 archivos
        all_exist = out_train.exists() and out_valid.exists() and out_test.exists()

        if all_exist:
            print(f"[OK] Ventanas ya existen. Cargando desde disco: {out_dir_L}")
            X_train, y_train = _load_npz(out_train)
            X_valid, y_valid = _load_npz(out_valid)
            X_test,  y_test  = _load_npz(out_test)
        else:
            missing = [p.name for p in [out_train, out_valid, out_test] if not p.exists()]
            print(f"[BUILD] Faltan {missing}. Generando y guardando (npz_compressed, X=float16, y=float32)...")

            # Esta función ya construye si no existen y guarda:
            # - np.savez_compressed(...)
            # - X float16 / y float32 (forzado en _save_npz)
            X_train, y_train, X_valid, y_valid, X_test, y_test = prepare_or_load_seq2one_windows_npz(
                mnq_train=df_tr,
                mnq_valid=df_va,
                mnq_test=df_te,
                features=features_to_windows,
                target_col=target_col,
                window_size=L,
                out_train=out_train,
                out_valid=out_valid,
                out_test=out_test,
                date_col="date",
                flatten=False,
                drop_windows_with_nan=True,
                verbose=True,
                repair_axis_order_if_needed=True,
                mmap_mode=None,
            )

        # Info compacta + tamaño de archivo + dtype + estimación de compresión
        xy_info_seq2one_compact(
            target_col=target_col,
            window_size_name=f"L{L}",
            horizon_min=int(target_col.split("_")[1]),
            X_train=X_train, y_train=y_train,
            X_valid=X_valid, y_valid=y_valid,
            X_test=X_test,   y_test=y_test,
            window_size=L,
            n_features=len(features_to_windows),
            path_train=out_train,
            path_valid=out_valid,
            path_test=out_test,
        )

        manifest = update_seq2one_manifest(
            manifest,
            target_col=target_col,
            window_size=L,
            n_features=len(features_to_windows),
            features=features_to_windows,
            X_train=X_train, y_train=y_train,
            X_valid=X_valid, y_valid=y_valid,
            X_test=X_test,   y_test=y_test,
            path_train=out_train,
            path_valid=out_valid,
            path_test=out_test,
            date_col="date",
            flatten=False,
        )

save_seq2one_manifest(
    manifest,
    out_path=OUT_WINDOWS_SEQ2ONE_DIR / "seq2one_windows_manifest.json",
)


SEQ2ONE | TARGET=delta_60 | L=30 | F=36
[BUILD] Faltan ['windows_delta_60_train.npz', 'windows_delta_60_valid.npz', 'windows_delta_60_test.npz']. Generando y guardando (npz_compressed, X=float16, y=float32)...
[delta_60 | L=30 | train] BUILD  X=(463872, 30, 36)  y=(463872,)  -> windows_delta_60_train.npz
[delta_60 | L=30 | valid] BUILD  X=(99328, 30, 36)  y=(99328,)  -> windows_delta_60_valid.npz
[delta_60 | L=30 | test] BUILD  X=(99840, 30, 36)  y=(99840,)  -> windows_delta_60_test.npz
[delta_60 | L=30 | train]           | X:(463872, 30, 36) (float16) | y:(463872,) (float32) | N=463872 | (L,F)=(30,36) | OK | file=26.00MB | raw~957.32MB | npz-compressed (probable) | ratio=0.027
[delta_60 | L=30 | valid]           | X:(99328, 30, 36) (float16) | y:(99328,) (float32) | N=99328 | (L,F)=(30,36) | OK | file=5.52MB | raw~204.99MB | npz-compressed (probable) | ratio=0.027
[delta_60 | L=30 | test ]           | X:(99840, 30, 36) (float16) | y:(99840,) (float32) | N=99840 | (L,F)=(30,36) | OK |

PosixPath('/content/drive/MyDrive/neural_profit/data/windows/seq2one/seq2one_windows_manifest.json')

## **4.3. Construcción de ventanas `seq2seq`**

### **4.3.1. Generador seq2seq**

In [89]:
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from typing import List, Tuple

def generate_windows_seq2seq(
    df: pd.DataFrame,
    *,
    date_col: str,
    features: List[str],
    target_col: str,
    window_size: int,
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Genera ventanas SEQ2SEQ (sliding intradía) agrupadas por `date_col`.

    Definición (por cada día)
    -------------------------
    Sea Xg = features del día con shape (n, F) y yg = target con shape (n,).

    Se generan ventanas deslizantes de longitud L = window_size:

      - Xw[t] = Xg[t : t+L, :]   -> entrada de longitud L
      - yw[t] = yg[t : t+L]      -> salida de longitud L

    Por lo tanto:
      - X: (N, L, F)   si flatten=False
      - X: (N, L*F)    si flatten=True
      - y: (N, L)      siempre

    Ahorro de espacio (para almacenamiento posterior)
    ------------------------------------------------
    - X se retorna en float16 para reducir tamaño (aprox. 50% vs float32).
    - y se mantiene en float32 para estabilidad numérica de targets y métricas.

    Notas importantes
    -----------------
    - El uso de `groupby(date_col)` evita crear ventanas que crucen el límite entre días
      (previene leakage cross-day).
    - `sliding_window_view` puede devolver Xw como (N, F, L) en algunas variantes; se normaliza
      a (N, L, F) con swapaxes si corresponde.
    - Esta función NO guarda a disco; solo genera arrays. Para NPZ comprimido, se usa
      `np.savez_compressed` en la función de guardado/caché.
    """
    # -------------------------
    # Validaciones básicas
    # -------------------------
    if window_size <= 0:
        raise ValueError("window_size debe ser un entero positivo.")

    # Verificar columnas requeridas
    required_cols = [date_col] + features + [target_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en df: {missing}")

    # Acumuladores por día para concatenar al final
    X_all: List[np.ndarray] = []
    y_all: List[np.ndarray] = []

    # Cantidad de features
    F = len(features)

    # -------------------------
    # Construcción por día
    # -------------------------
    for _, g in df.groupby(date_col, sort=False):
        # Reset index para usar índices [0..n-1] por día
        # sort_index() mantiene el orden temporal si el índice original ya estaba en orden
        g = g.sort_index().reset_index(drop=True)
        n = len(g)

        # Si el día no tiene suficientes filas, no genera ventanas
        if n < window_size:
            continue

        # -------------------------
        # Extracción a numpy
        # -------------------------
        # Xg se mantiene en float32 durante el armado por estabilidad
        Xg = g[features].to_numpy(dtype=np.float32, copy=False)   # (n, F)

        # yg en float32 (target)
        yg = g[target_col].to_numpy(dtype=np.float32, copy=False) # (n,)

        # -------------------------
        # Ventanas deslizantes para X
        # -------------------------
        # Ideal: (n-L+1, L, F)
        Xw = sliding_window_view(Xg, window_shape=window_size, axis=0)

        # Validación ndim
        if Xw.ndim != 3:
            raise ValueError(f"Xw ndim inesperado: {Xw.ndim} | shape={Xw.shape}")

        # Normalización por compatibilidad: si viene (N, F, L) -> (N, L, F)
        if Xw.shape[1] == F and Xw.shape[2] == window_size:
            Xw = np.swapaxes(Xw, 1, 2)

        # Validación final de shape
        if not (Xw.shape[1] == window_size and Xw.shape[2] == F):
            raise ValueError(f"Xw shape inválido: {Xw.shape} (esperado: (N,{window_size},{F}))")

        # -------------------------
        # Ventanas deslizantes para y
        # -------------------------
        # yw: (n-L+1, L)
        yw = sliding_window_view(yg, window_shape=window_size, axis=0)

        if yw.ndim != 2 or yw.shape[1] != window_size:
            raise ValueError(f"yw shape inválido: {yw.shape} (esperado: (N,{window_size}))")

        # -------------------------
        # Filtrado de NaNs (opcional)
        # -------------------------
        if drop_windows_with_nan:
            # X válido si no hay NaNs en ninguna posición de (L,F)
            x_ok = ~np.isnan(Xw).any(axis=(1, 2))
            # y válido si no hay NaNs en ninguna posición de la secuencia
            y_ok = ~np.isnan(yw).any(axis=1)
            ok = x_ok & y_ok
            Xw = Xw[ok]
            yw = yw[ok]

        if Xw.size == 0:
            continue

        # -------------------------
        # Aplanado (opcional)
        # -------------------------
        # Para modelos que consumen 2D: (N, L, F) -> (N, L*F)
        if flatten:
            Xw = Xw.reshape(Xw.shape[0], -1)

        # -------------------------
        # Cast final de dtypes (alineado a su almacenamiento)
        # -------------------------
        # X en float16 (ahorro), y en float32 (estabilidad)
        X_all.append(np.asarray(Xw, dtype=np.float16))
        y_all.append(np.asarray(yw, dtype=np.float32))

    # -------------------------
    # Caso borde: sin ventanas
    # -------------------------
    if not X_all:
        if flatten:
            X = np.empty((0, window_size * F), dtype=np.float16)
        else:
            X = np.empty((0, window_size, F), dtype=np.float16)
        y = np.empty((0, window_size), dtype=np.float32)
        return X, y

    # -------------------------
    # Concatenación final
    # -------------------------
    X = np.concatenate(X_all, axis=0)
    y = np.concatenate(y_all, axis=0)

    return X, y


### **4.3.2. Load/Build para SEQ2SEQ**

In [90]:
import numpy as np
from pathlib import Path
from typing import List, Tuple

def prepare_or_load_seq2seq_windows_npz(
    *,
    mnq_train,
    mnq_valid,
    mnq_test,
    features: List[str],
    target_col: str,
    window_size: int,

    # 1 path por split (npz con X e y)
    out_train,
    out_valid,
    out_test,

    date_col: str = "date",
    flatten: bool = False,
    drop_windows_with_nan: bool = True,
    verbose: bool = True,

    # Nota: en .npz no hay mmap real como en .npy; se deja por compatibilidad
    mmap_mode: str | None = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    SEQ2SEQ sliding intradía:
      - Genera o carga ventanas deslizantes por día:
          X: (N, L, F)  o (N, L*F) si flatten=True
          y: (N, L)
      - Guarda/carga en un único .npz por split (claves: 'X' y 'y')
      - Guardado alineado a su estrategia de espacio:
          X -> float16
          y -> float32
          np.savez_compressed(...)
    """

    def _to_path(p) -> Path:
        return p if isinstance(p, Path) else Path(str(p))

    def _ensure_parent_dir(path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)

    def _save_npz(path_npz: Path, X: np.ndarray, y: np.ndarray) -> None:
        """
        Guarda X e y en un .npz comprimido, forzando dtypes:
          - X float16 (ahorro)
          - y float32 (estabilidad)
        """
        _ensure_parent_dir(path_npz)
        X = np.asarray(X, dtype=np.float16)
        y = np.asarray(y, dtype=np.float32)
        np.savez_compressed(path_npz, X=X, y=y)

    def _load_npz(path_npz: Path):
        """
        Carga X e y desde un .npz.
        - allow_pickle=False por seguridad.
        - mmap_mode no aplica realmente a .npz (se mantiene por compatibilidad).
        """
        with np.load(path_npz, allow_pickle=False) as z:
            X = z["X"]
            y = z["y"]
        return X, y

    def _load_or_build(split_name: str, df, path_npz):
        path_npz = _to_path(path_npz)

        action = "load"
        if not path_npz.exists():
            action = "build"
            X, y = generate_windows_seq2seq(
                df=df,
                date_col=date_col,
                features=features,
                target_col=target_col,
                window_size=window_size,
                flatten=flatten,
                drop_windows_with_nan=drop_windows_with_nan,
            )
            _save_npz(path_npz, X, y)
        else:
            X, y = _load_npz(path_npz)

        # Normalización defensiva de dtype (por si el archivo venía de otra corrida)
        if X.dtype != np.float16:
            X = np.asarray(X, dtype=np.float16)
        if y.dtype != np.float32:
            y = np.asarray(y, dtype=np.float32)

        if verbose:
            print(
                f"[{target_col} | L={window_size} | {split_name}] "
                f"{action.upper()}  X={X.shape}  y={y.shape}  -> {path_npz.name}"
            )

        return X, y

    X_train, y_train = _load_or_build("train", mnq_train, out_train)
    X_valid, y_valid = _load_or_build("valid", mnq_valid, out_valid)
    X_test,  y_test  = _load_or_build("test",  mnq_test,  out_test)

    return X_train, y_train, X_valid, y_valid, X_test, y_test

### **4.3.3. Info para SEQ2SEQ**

In [91]:
from pathlib import Path
import numpy as np

def xy_info_seq2seq_compact(
    target_col,
    window_size_name,
    horizon_min: int,
    X_train, y_train,
    X_valid, y_valid,
    X_test,  y_test,
    *,
    window_size: int,
    n_features: int,
    # Opcional: rutas a los .npz (para mostrar formato y tamaño)
    path_train: str | Path | None = None,
    path_valid: str | Path | None = None,
    path_test:  str | Path | None = None,
):
    """
    Muestra información estructural compacta por split (SEQ2SEQ sliding), incluyendo:
    - shape y validaciones de dimensiones
    - dtype de X e y
    - (opcional) tamaño de archivo y estimación de "npz-compressed" vs "npz"

    Esperado:
      - X: (N, L, F) o (N, L*F) si flatten=True
      - y: (N, L)   (o (N, L, 1) si se eligiera esa representación)
    """

    def _to_path(p):
        return None if p is None else (p if isinstance(p, Path) else Path(str(p)))

    def _human_bytes(n: int) -> str:
        units = ["B", "KB", "MB", "GB", "TB"]
        x = float(n)
        for u in units:
            if x < 1024.0 or u == units[-1]:
                return f"{x:.2f}{u}"
            x /= 1024.0
        return f"{x:.2f}TB"

    def _file_info(path: Path | None, X: np.ndarray, y: np.ndarray) -> str:
        if path is None or not path.exists():
            return "file=NA"

        size_disk = path.stat().st_size

        # Estimación "bruta" sin headers
        raw_est = int(getattr(X, "nbytes", 0) + getattr(y, "nbytes", 0))
        ratio = (size_disk / raw_est) if raw_est > 0 else float("nan")

        if raw_est > 0 and ratio < 0.85:
            kind = "npz-compressed (probable)"
        else:
            kind = "npz (sin compresión o compresión mínima)"

        return f"file={_human_bytes(size_disk)} | raw~{_human_bytes(raw_est)} | {kind} | ratio={ratio:.3f}"

    p_train = _to_path(path_train)
    p_valid = _to_path(path_valid)
    p_test  = _to_path(path_test)

    def _print_line(name, X, y, path: Path | None):
        if not hasattr(X, "shape") or not hasattr(y, "shape"):
            print(f"{name} | ERROR: X o y sin shape")
            return

        N = X.shape[0] if X.ndim >= 1 else 0

        # --------- X checks ----------
        x_ok = False
        x_dim_info = "dim=?"

        if X.ndim == 3:
            L, F = X.shape[1], X.shape[2]
            x_ok = (L == window_size) and (F == n_features)
            x_dim_info = f"(L,F)=({L},{F})"
        elif X.ndim == 2:
            dim = X.shape[1]
            expected = window_size * n_features
            x_ok = (dim == expected)
            x_dim_info = f"dim={dim}"
        else:
            x_ok = False

        # --------- y checks ----------
        y_ok = False
        y_dim_info = "dim=?"

        if y.ndim == 2:
            Ly = y.shape[1]
            y_ok = (y.shape[0] == N) and (Ly == window_size)
            y_dim_info = f"L={Ly}"
        elif y.ndim == 3 and y.shape[2] == 1:
            Ly = y.shape[1]
            y_ok = (y.shape[0] == N) and (Ly == window_size)
            y_dim_info = f"L={Ly},C=1"
        else:
            y_ok = False

        status = "OK" if (x_ok and y_ok) else "ERROR"

        # dtype info
        x_dtype = getattr(X, "dtype", "NA")
        y_dtype = getattr(y, "dtype", "NA")

        # file info
        finfo = _file_info(path, X, y)

        print(
            f"{name:<35} | "
            f"X:{X.shape} ({x_dtype}) | "
            f"y:{y.shape} ({y_dtype}) | "
            f"N={N} | "
            f"{x_dim_info} | "
            f"{y_dim_info} | "
            f"{status} | "
            f"{finfo}"
        )

    _print_line(f"[{target_col} | L={window_size} | train]", X_train, y_train, p_train)
    _print_line(f"[{target_col} | L={window_size} | valid]", X_valid, y_valid, p_valid)
    _print_line(f"[{target_col} | L={window_size} | test ]", X_test,  y_test,  p_test)
    print()


### **4.3.4. Utilidades**

In [92]:
OUT_WINDOWS_SEQ2SEQ_DIR = DRIVE_DIR / "data/windows/seq2seq"

### **4.3.5. Generación de resumen**

In [93]:
# =========================
# 2) Builder SEQ2SEQ
# =========================

def update_seq2seq_manifest(
    manifest: dict,
    *,
    target_col: str,
    window_size: int,
    n_features: int,
    features: list[str],
    X_train, y_train, X_valid, y_valid, X_test, y_test,
    path_train: Path,
    path_valid: Path,
    path_test: Path,
    date_col: str = "date",
    flatten: bool = False,
) -> dict:
    """
    Actualiza el manifest global con la entrada (target_col, window_size) para SEQ2SEQ.
    Guarda información por split y deja trazabilidad de shapes/dtypes y tamaños.
    """
    key = f"{target_col}__L{window_size}"

    # Validación ligera (opcional pero útil): y debe tener L en eje 1
    def _y_has_L(y) -> bool:
        return (hasattr(y, "ndim") and ((y.ndim == 2 and y.shape[1] == window_size) or (y.ndim == 3 and y.shape[1] == window_size)))

    manifest["items"][key] = {
        "mode": "seq2seq",
        "target": target_col,
        "window_size": int(window_size),
        "n_features": int(n_features),
        "features": list(features),
        "date_col": date_col,
        "flatten": bool(flatten),
        "format_expected": "npz_compressed",
        "checks": {
            "y_train_has_L": bool(_y_has_L(y_train)),
            "y_valid_has_L": bool(_y_has_L(y_valid)),
            "y_test_has_L": bool(_y_has_L(y_test)),
        },
        "splits": {
            "train": _split_file_info(Path(path_train), np.asarray(X_train), np.asarray(y_train)),
            "valid": _split_file_info(Path(path_valid), np.asarray(X_valid), np.asarray(y_valid)),
            "test":  _split_file_info(Path(path_test),  np.asarray(X_test),  np.asarray(y_test)),
        },
    }

    return manifest

### **4.3.6. Generación de ventanas `seq2seq`**

In [95]:
from datetime import datetime
from pathlib import Path
import numpy as np

# =========================
# MANIFEST SEQ2SEQ (init)
# =========================
seq2seq_manifest = {
    "schema": "seq2seq_windows_manifest_v1",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "items": {}
}

for target_col, (df_tr, df_va, df_te) in targets_windows.items():
    for L in window_sizes:

        print("\n" + "=" * 70)
        print(f"SEQ2SEQ | TARGET={target_col} | L={L} | F={n_features}")
        print("=" * 70)

        out_dir_L = OUT_WINDOWS_SEQ2SEQ_DIR / f"L{L}"
        out_dir_L.mkdir(parents=True, exist_ok=True)

        out_train = out_dir_L / f"windows_{target_col}_train.npz"
        out_valid = out_dir_L / f"windows_{target_col}_valid.npz"
        out_test  = out_dir_L / f"windows_{target_col}_test.npz"

        all_exist = out_train.exists() and out_valid.exists() and out_test.exists()

        if all_exist:
            print(f"[OK] Ventanas ya existen. Cargando desde disco: {out_dir_L}")
            X_train, y_train = _load_npz(out_train)
            X_valid, y_valid = _load_npz(out_valid)
            X_test,  y_test  = _load_npz(out_test)
        else:
            missing = [p.name for p in [out_train, out_valid, out_test] if not p.exists()]
            print(f"[BUILD] Faltan {missing}. Generando y guardando (npz_compressed, X=float16, y=float32)...")

            X_train, y_train, X_valid, y_valid, X_test, y_test = prepare_or_load_seq2seq_windows_npz(
                mnq_train=df_tr,
                mnq_valid=df_va,
                mnq_test=df_te,
                features=features_to_windows,
                target_col=target_col,
                window_size=L,
                out_train=out_train,
                out_valid=out_valid,
                out_test=out_test,
                date_col="date",
                flatten=False,
                drop_windows_with_nan=True,
                verbose=True,
                mmap_mode=None,
            )

        # Info compacta (con file size + dtype)
        xy_info_seq2seq_compact(
            target_col=target_col,
            window_size_name=f"L{L}",
            horizon_min=int(target_col.split("_")[1]),
            X_train=X_train, y_train=y_train,
            X_valid=X_valid, y_valid=y_valid,
            X_test=X_test,   y_test=y_test,
            window_size=L,
            n_features=len(features_to_windows),
            path_train=out_train,
            path_valid=out_valid,
            path_test=out_test,
        )

        # =========================
        # UPDATE MANIFEST (SEQ2SEQ)
        # =========================
        update_seq2seq_manifest(
            seq2seq_manifest,
            target_col=target_col,
            window_size=L,
            n_features=len(features_to_windows),
            features=features_to_windows,
            X_train=X_train, y_train=y_train,
            X_valid=X_valid, y_valid=y_valid,
            X_test=X_test,   y_test=y_test,
            path_train=out_train,
            path_valid=out_valid,
            path_test=out_test,
            date_col="date",
            flatten=False,
        )

# =========================
# SAVE MANIFEST (END)
# =========================
save_seq2one_manifest(
    seq2seq_manifest,
    out_path=OUT_WINDOWS_SEQ2SEQ_DIR / "seq2seq_windows_manifest.json",
)


SEQ2SEQ | TARGET=delta_60 | L=30 | F=36
[OK] Ventanas ya existen. Cargando desde disco: /content/drive/MyDrive/neural_profit/data/windows/seq2seq/L30
[delta_60 | L=30 | train]           | X:(463872, 30, 36) (float16) | y:(463872, 30) (float32) | N=463872 | (L,F)=(30,36) | L=30 | OK | file=26.75MB | raw~1008.63MB | npz-compressed (probable) | ratio=0.027
[delta_60 | L=30 | valid]           | X:(99328, 30, 36) (float16) | y:(99328, 30) (float32) | N=99328 | (L,F)=(30,36) | L=30 | OK | file=5.68MB | raw~215.98MB | npz-compressed (probable) | ratio=0.026
[delta_60 | L=30 | test ]           | X:(99840, 30, 36) (float16) | y:(99840, 30) (float32) | N=99840 | (L,F)=(30,36) | L=30 | OK | file=5.74MB | raw~217.09MB | npz-compressed (probable) | ratio=0.026


SEQ2SEQ | TARGET=delta_60 | L=60 | F=36
[OK] Ventanas ya existen. Cargando desde disco: /content/drive/MyDrive/neural_profit/data/windows/seq2seq/L60
[delta_60 | L=60 | train]           | X:(436692, 60, 36) (float16) | y:(436692, 60) (floa

PosixPath('/content/drive/MyDrive/neural_profit/data/windows/seq2seq/seq2seq_windows_manifest.json')

# **5. Alineamiento con libro de ML**

## Preprocesamiento y escalamiento de datos

Este paso se realiza **después del split temporal** y **antes del entrenamiento del modelo**, de acuerdo con las buenas prácticas de *Machine Learning* para series temporales financieras.

---

### Aspectos correctamente alineados

1. **Orden del proceso**
   - El split temporal se realiza **antes** del escalamiento.
   - El escalamiento se realiza **antes** del entrenamiento del modelo.

2. **Regla crítica anti-leakage**
   - El *scaler* se ajusta **exclusivamente con el conjunto de entrenamiento (TRAIN)**.
   - Los conjuntos de *validation* y *test* se transforman utilizando ese mismo *scaler*,
     sin volver a ajustarlo.

3. **Escalamiento aplicado únicamente a features**
   - Las variables de entrada (OHLCV / features) son escaladas.
   - El target (`delta_pts_H`) **no se escala**, preservando su interpretación económica en unidades de puntos.

4. **Consistencia entre horizontes**
   - Se utilizan *scalers* independientes para **H = 60** y **H = 90**.
   - No se mezclan estadísticas entre distintos horizontes de predicción.

5. **Persistencia**
   - Los *scalers* se guardan para asegurar reproducibilidad.
   - Son reutilizables en etapas posteriores de inferencia.

---

### Criterio de escalamiento

Se utiliza **StandardScaler (z-score)**, ajustado exclusivamente con el conjunto de
entrenamiento, por su adecuación a modelos sensibles a la escala.

El escalamiento se realiza **por feature de forma global**, y no de manera independiente
por jornada bursátil.

---

### Verificación automática del escalamiento

Se realizó un *sanity check* sobre el conjunto de entrenamiento escalado, verificando que:

- la media por feature sea aproximadamente 0,
- el desvío estándar por feature sea aproximadamente 1,

únicamente en el conjunto **TRAIN**.

Resultado para **H = 60**:

[OK] Escalamiento TRAIN verificado |

max |mean| = 8.2516e-16

max |std-1| = 8.8818e-15

Estos valores corresponden a error numérico de precisión flotante, confirmando que el
escalamiento fue correctamente ajustado sobre el conjunto de entrenamiento, sin
contaminación de *validation* ni *test*.
